# 04 · Protección de datos y ética (R6)
Parte del DW que deja `03_dw` (`data/dw/covid_dw.zip`). 

In [1]:
import sqlite3
import zipfile
import pandas as pd

# El DW se sube comprimido (pesa ~130 MB): primero se descomprime
zipfile.ZipFile("../data/dw/covid_dw.zip").extractall("../data/dw")
dw = sqlite3.connect("../data/dw/covid_dw.db")

territorios = pd.read_sql("SELECT * FROM dim_territorio", dw)
print("Territorios:", len(territorios))
territorios.groupby("nivel").size()

Territorios: 4280


nivel
condado    3318
pais        196
region      766
dtype: int64

## 1 · Argumento legal 
**Pregunta:** los datos de Johns Hopkins, ¿son datos personales según la Ley 19.628 y la Ley 21.719?

### Los conceptos
- **Dato personal:** cualquier información vinculada a una persona natural **identificada o identificable**
  (Ley 19.628, art. 2 letra f; la Ley 21.719 mantiene la idea y agrega que se puede identificar *directa o
  indirectamente*, por ejemplo cruzando con otra fuente).
- **Dato sensible:** entre otros, los **estados de salud** (art. 2 letra g). Tienen el estándar de protección más alto:
  como regla general no se pueden tratar, salvo excepciones (ley, consentimiento, salud).
- **Dato estadístico:** el que, en su origen o por su tratamiento, **no puede asociarse a un titular identificado o
  identificable** (art. 2 letra e). Ese es el caso general de este proyecto.
- La **Ley 21.719** (2024) modifica y reemplaza buena parte de la 19.628 desde el **1-dic-2026**: crea la Agencia de
  Protección de Datos Personales, multas de hasta 20.000 UTM y obligación de notificar brechas. El proyecto se entrega
  antes de esa fecha, pero el tablero se usaría después: hay que cumplir con las dos.

### ¿Son datos personales? 
En general, **no**
1. Son **conteos agregados**: casos y fallecidos acumulados por territorio y día. No hay nombre, RUT, edad, sexo
   ni dirección; ninguna fila corresponde a una persona.
2. El territorio más chico es un **condado de EE.UU.** o una **región de Chile**: miles o millones de habitantes detrás
   de cada número.
3. Los publicaron los gobiernos y Johns Hopkins **ya agregados**; el proyecto nunca tuvo microdatos (fichas de pacientes).
4. Base de licitud de quien los generó: **interés público** (vigilancia epidemiológica, como el registro de vacunación
   del Minsal.

### ¿Cuándo **podrían** serlo? 
Cuando la celda es pequeña

La palabra clave es **identificable**. El laboratorio observamos que borrar el nombre no basta: el 99,5 % de los
pacientes quedaba solo en su combinación de fecha, sexo y comuna, y un `merge` con otra lista lo reidentificaba.
Aquí pasa lo mismo cuando se juntan:
- un **territorio con poca población** (en el DW hay condados con 86 habitantes y territorios con 67),
- un **conteo de 1 a 4** casos o fallecidos,
- y una **fecha exacta** con **latitud y longitud**.

Con eso, alguien del lugar, la prensa local o una red social puede saber quién es la persona. Y lo que se revela es
**dato de salud, o sea sensible**: que tuvo COVID o que murió por COVID. Pasó con los primeros casos de cada país o
región, que fueron noticia con edad, ciudad y viaje.

**Conclusión:** los datos son estadísticos y se pueden publicar, **salvo las celdas pequeñas**, que podrían volver
identificable a una persona y exponer un dato sensible. Por eso el tablero aplica una regla de supresión.

## 2 · Supresión de celdas pequeñas 
**Regla:** en territorios con **menos de 100.000 habitantes**, los casos o fallecidos nuevos de un día **entre 1 y 4**
no se muestran (quedan vacíos). El 0 sí se muestra: no revela a nadie.

- Se aplica a países y regiones (lo que muestra el tablero). Las regiones de Chile tienen todas más de 100.000
  habitantes, así que en Chile no se suprime nada.
- Territorios **sin población** en la tabla UID (170 regiones, sobre todo de los primeros meses) no se pueden evaluar
  y quedan sin suprimir.

In [2]:
def suprimir(df, umbral_poblacion=100_000):
    """Deja vacíos los casos/fallecidos nuevos de 1 a 4 en territorios con menos de umbral_poblacion habitantes."""
    df = df.copy()
    poca_poblacion = df["poblacion"] < umbral_poblacion
    for col in ["casos_nuevos", "fallecidos_nuevos"]:
        celda_pequena = poca_poblacion & df[col].between(1, 4)
        df.loc[celda_pequena, col] = None
    return df

### Territorios con poca población
Países y regiones bajo 100.000 habitantes: islas, microestados y regiones remotas.

In [3]:
pequenos = territorios[(territorios["nivel"] != "condado") & (territorios["poblacion"] < 100_000)]
print("Países y regiones con menos de 100.000 habitantes:", len(pequenos),
      "| países:", (pequenos["nivel"] == "pais").sum(), "| regiones:", (pequenos["nivel"] == "region").sum())
pequenos.sort_values("poblacion")[["pais", "region_estado", "nivel", "poblacion"]].head(10)

Países y regiones con menos de 100.000 habitantes: 52 | países: 13 | regiones: 39


,pais,region_estado,nivel,poblacion
4265,United Kingdom,Pitcairn Islands,region,67.0
266,Holy See,,pais,809.0
496,New Zealand,Niue,region,1650.0
4258,United Kingdom,Falkland Islands (Malvinas),region,3483.0
4263,United Kingdom,Montserrat,region,4999.0
4266,United Kingdom,"Saint Helena, Ascension and Tristan da Cunha",region,5661.0
232,France,Saint Pierre and Miquelon,region,5795.0
230,France,Saint Barthelemy,region,9885.0
473,Nauru,,pais,10834.0
709,Tuvalu,,pais,11792.0


### Aplicar la regla y medir
Se leen del DW los nuevos diarios de todos los países y regiones con su población, se aplica `suprimir` y se compara.

In [4]:
PAISES_REGIONES = """
    SELECT t.poblacion, f.casos_nuevos, f.fallecidos_nuevos
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    WHERE t.nivel IN ('pais', 'region')
"""
antes = pd.read_sql(PAISES_REGIONES, dw)
despues = suprimir(antes)

print("Filas país/región × día:", len(antes))
for col in ["casos_nuevos", "fallecidos_nuevos"]:
    print(col, "· celdas suprimidas:", (antes[col].notna() & despues[col].isna()).sum())

Filas país/región × día: 771114
casos_nuevos · celdas suprimidas: 5421
fallecidos_nuevos · celdas suprimidas: 2054


## 3 · Advertencias de interpretación 
### 3.1 · La letalidad aparente depende del testeo
La **letalidad aparente** = fallecidos ÷ casos **confirmados**. No es la probabilidad de morir si uno se contagia:
el denominador son solo los contagios que **se detectaron**, y eso depende de cuánto se testeó.

Chile por año (con acumulados: último día del año − último día del año anterior):

In [5]:
CHILE_DIA = """
    SELECT d.fecha, d.anio,
           SUM(f.casos_acumulados) AS casos, SUM(f.fallecidos_acumulados) AS fallecidos
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE t.pais = 'Chile'
    GROUP BY d.fecha
    ORDER BY d.fecha
"""
chile_dia = pd.read_sql(CHILE_DIA, dw)

chile_anio = chile_dia.groupby("anio")[["casos", "fallecidos"]].last()
chile_anio = chile_anio.diff().fillna(chile_anio)          # casos y fallecidos de cada año
chile_anio["letalidad_%"] = (chile_anio["fallecidos"] / chile_anio["casos"] * 100).round(2)
chile_anio

,casos,fallecidos,letalidad_%
anio,,,
2020,608973.0,16608.0,2.73
2021,1197521.0,22507.0,1.88
2022,3231033.0,24057.0,0.74
2023,154759.0,1101.0,0.71


La letalidad aparente de Chile cae de **2,7 % en 2020 a 0,7 % en 2022**. Parte es la vacunación (P3), pero no se
puede atribuir todo a ella:
- **En 2020 se testeaba poco** y sobre todo a quien tenía síntomas: se detectaban los casos graves y los leves
  quedaban fuera del denominador → letalidad inflada.
- **En 2022 se testeaba mucho más** (test rápidos, búsqueda activa) y ómicron daba muchos casos leves → el
  denominador crece y la letalidad baja.
- Además los **fallecidos llegan con rezago** respecto de los casos: al inicio de una ola la letalidad se ve baja y al
  final se ve alta.
- En 2022 se empezaron a usar **test caseros**, que no siempre se informan → vuelve a faltar denominador.

Nuestras fuentes **no traen el número de test**, así que no se puede corregir la letalidad por testeo: se informa
como "aparente" y con esta advertencia.

### 3.2 · Comparar países exige cautela
Letalidad aparente acumulada al 9-mar-2023 en países con 1 millón de habitantes o más :

In [6]:
PAISES_FINAL = """
    SELECT t.pais, SUM(f.casos_acumulados) AS casos, SUM(f.fallecidos_acumulados) AS fallecidos,
           MAX(t.poblacion_pais) AS poblacion, MAX(t.pct_65_mas) AS pct_65_mas
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    WHERE f.fecha_key = 20230309
    GROUP BY t.pais
"""
paises = pd.read_sql(PAISES_FINAL, dw)
paises = paises[paises["poblacion"] >= 1_000_000].copy()
paises["casos_por_100k"] = (paises["casos"] / paises["poblacion"] * 100_000).round(0)
paises["letalidad_%"] = (paises["fallecidos"] / paises["casos"] * 100).round(2)
paises = paises.sort_values("letalidad_%", ascending=False).reset_index(drop=True)

print("Países:", len(paises))
paises.head(6)

Países: 157


,pais,casos,fallecidos,poblacion,pct_65_mas,casos_por_100k,letalidad_%
0,"Korea, North",1.0,6.0,25778815.0,11.282498,0.0,600.00
1,Yemen,11945.0,2159.0,29825968.0,2.518882,40.0,18.07
2,Sudan,63829.0,5017.0,43849269.0,3.053637,146.0,7.86
3,Syria,57467.0,3164.0,17500657.0,4.618468,328.0,5.51
4,Somalia,27324.0,1361.0,15893219.0,2.537097,172.0,4.98
5,Peru,4487553.0,219539.0,32971846.0,8.754092,13610.0,4.89


- **Distinto testeo.** Los países con letalidad más alta (Yemen, Sudán, Siria, Somalia) son los que **menos casos
  detectaron por 100.000** habitantes: con pocos test solo se confirman los graves. Corea del Norte informó
  1 caso y 6 fallecidos: la cifra no significa nada.
- **Distinta definición de fallecido COVID.** Algunos países cuentan solo con test positivo; otros incluyen
  sospechosos o cualquier muerte con COVID. 
- **Distinta calidad del dato.** Retrocesos, días sin reporte y picos de rezago varían mucho entre países
  (índice de calidad : Colombia 68,3, Chile 96,6).
- **Distinta población.** Japón tiene 29 % de mayores de 65 y aun así una de las letalidades más bajas; África tiene
  poblaciones jóvenes. La edad, el ingreso y el acceso a salud influyen y no están en la letalidad.
- **Distinto momento.** Un país cuya ola grande fue con ómicron y con vacunas no se compara con uno que la tuvo en 2020.

## 4 · Supresión en condados de EE.UU.
El enunciado dice que **317 condados tienen menos de 5.000 habitantes**. Se aplica la misma regla (conteos de 1 a 4)
con ese umbral y se mide cuánta información se pierde.

In [7]:
condados_dw = territorios[(territorios["pais"] == "US") & (territorios["nivel"] == "condado")
                          & (territorios["poblacion"] < 5_000)]
print("Condados de EE.UU. con menos de 5.000 habitantes en el DW:", len(condados_dw))

Condados de EE.UU. con menos de 5.000 habitantes en el DW: 316


Los 317 de UID son 316 en el DW: `Yakutat plus Hoonah-Angoon` (Alaska) no aparece en los reportes con ese nombre
(JHU informa `Alaska/Yakutat`).

Los más chicos:

In [8]:
condados_dw.sort_values("poblacion")[["region_estado", "condado", "poblacion"]].head(10)

,region_estado,condado,poblacion
1335,Hawaii,Kalawao,86.0
3701,Texas,Loving,169.0
3685,Texas,King,272.0
3681,Texas,Kenedy,404.0
2516,Nebraska,Arthur,463.0
2518,Nebraska,Blaine,465.0
2486,Montana,Petroleum,487.0
2573,Nebraska,McPherson,494.0
2551,Nebraska,Grant,623.0
2677,New Mexico,Harding,625.0


In [9]:
def perdida(antes, despues):
    """Cuánta información se pierde al suprimir: celdas y personas que dejan de verse."""
    filas = []
    for col in ["casos_nuevos", "fallecidos_nuevos"]:
        suprimidas = antes[col].notna() & despues[col].isna()
        filas.append({
            "medida": col,
            "días con dato > 0": (antes[col] > 0).sum(),
            "celdas suprimidas": suprimidas.sum(),
            "% de días con dato": round(suprimidas.sum() / (antes[col] > 0).sum() * 100, 1),
            "personas ocultas": int(antes.loc[suprimidas, col].sum()),
            "% del total": round(antes.loc[suprimidas, col].sum() / antes[col].sum() * 100, 2),
            "territorios afectados": antes.loc[suprimidas, "territorio_key"].nunique(),
        })
    return pd.DataFrame(filas)

In [10]:
CONDADOS_US = """
    SELECT f.territorio_key, t.poblacion, f.casos_nuevos, f.fallecidos_nuevos
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    WHERE t.pais = 'US' AND t.nivel = 'condado'
"""
condados = pd.read_sql(CONDADOS_US, dw)
condados_suprimidos = suprimir(condados, umbral_poblacion=5_000)

print("Filas condado × día de EE.UU.:", len(condados))
perdida(condados, condados_suprimidos)

Filas condado × día de EE.UU.: 3505363


,medida,días con dato > 0,celdas suprimidas,% de días con dato,personas ocultas,% del total,territorios afectados
0,casos_nuevos,1897553,61504,3.2,109864,0.11,310
1,fallecidos_nuevos,407307,3515,0.9,3943,0.34,299


La misma medida, pero solo dentro de los 316 condados chicos (lo que deja de ver quien mira uno de ellos):

In [11]:
chicos = condados["poblacion"] < 5_000
perdida(condados[chicos], condados_suprimidos[chicos])

,medida,días con dato > 0,celdas suprimidas,% de días con dato,personas ocultas,% del total,territorios afectados
0,casos_nuevos,76164,61504,80.8,109864,45.22,310
1,fallecidos_nuevos,3546,3515,99.1,3943,98.90,299


### Cuánta información se pierde
- **Para EE.UU. completo, casi nada:** se oculta alrededor del 0,1 % de los casos y el 0,3 % de los fallecidos
  de los condados. Las tendencias nacionales y estatales no cambian.
- **Para cada condado chico, casi todo:** se oculta cerca de la mitad de sus casos y prácticamente todos sus
  fallecidos (llegan de a 1 o 2). Esa es la información que podía identificar a alguien, así que la pérdida es
  el precio de la protección.

In [12]:
dw.close()